In [4]:
import os
import glob
import pandas as pd
import shutil
from tqdm import tqdm
from pathlib import Path
import shutil

In [ ]:
# Pandas :: view ALL
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

In [ ]:
# Pandas :: Default view
pd.reset_option('display.max_rows')
pd.reset_option('display.max_columns')
pd.reset_option('display.width')
pd.reset_option('display.max_colwidth')

In [5]:
base   = r"D:\JHAhn\8641_conductivity\DTI_ALPS\1028"
RT_dir = os.path.join(base, "8641_Dem+MCI+AD")

# 1
xlsx_in = r"D:\JHAhn\files\안재혁_8641_Auto_ALPS_index_19-24_main+CTI.xlsx"
USED = os.path.join(RT_dir, "USED")

# 2

# 3


In [6]:
# Prepare 'tqdm'
try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, *a, **k): return x

C:\Users\user\anaconda3\envs\JHA_dev\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Prepare USED individuals as each Folders in :

In [8]:
OVERWRITE_IF_EXISTS = False
DRY_RUN = False

df = pd.read_excel(xlsx_in, usecols=["PDI"])
pdis = (
    df["PDI"]
    .astype(str)
    .str.strip()
    .replace({"": pd.NA})
    .dropna()
    .drop_duplicates()
    .tolist()
)
df

,PDI
0,ADEPT54_HCS_NODDI24
1,EPT2022AD01_KDKM_NODDI32
2,EPT2022AD02_SKM_NODDI32
3,EPT2022AD03_KKS_NODDI32
4,EPT2022AD05_KIJD_NODDI32
...,...
97,MWI_CN79_CYE_NODDI32
98,MWI_CN80_LSR_NODDI32
99,MWI_CN81_CSH_NODDI32
100,MWI_CN82_LMR_NODDI32


In [9]:
os.makedirs(USED, exist_ok=True)

In [10]:
moved, skipped_exists, not_found, errors = [], [], [], []

for name in tqdm(pdis, desc="Moving selected subjects", unit="person"):
    src = os.path.join(RT_dir, name)
    if not os.path.isdir(src):
        not_found.append(name)
        continue

    dst = os.path.join(USED, name)
    if os.path.exists(dst):
        if OVERWRITE_IF_EXISTS:
            # 기존 대상 삭제 후 이동 (주의!)
            try:
                if not DRY_RUN:
                    shutil.rmtree(dst)
            except Exception as e:
                errors.append((name, f"remove dst failed: {e}"))
                continue
        else:
            skipped_exists.append(name)
            continue

    try:
        if not DRY_RUN:
            shutil.move(src, dst)
        moved.append(name)
    except Exception as e:
        errors.append((name, str(e)))

# Printing
total = len(pdis)
matched_in_source = len(moved) + len(skipped_exists)  # 소스에 폴더 존재했던 수
moved_ratio = f"{len(moved)}/{total} ({(len(moved)/total*100):.1f}%)"
matched_ratio = f"{matched_in_source}/{total} ({(matched_in_source/total*100):.1f}%)"

print("\n=== SUMMARY ===")
print(f"Total in Excel (unique PDI): {total}")
print(f"Matched in source: {matched_ratio}")          # 소스에 폴더 있었던 수(이동+이미 USED)
print(f"Matched & moved this run: {moved_ratio}")     # 이번 실행에서 실제 이동된 수
print(f"Already in USED (skipped): {len(skipped_exists)}")
print(f"Not found in source: {len(not_found)}")
print(f"Errors: {len(errors)}")
if DRY_RUN:
    print("\n[DRY RUN] !!!! Movement NOT EXECUTED !!!!")

# --- 로그 저장 (base에 저장) ---
log_path = os.path.join(base, "USED_move_log.xlsx")
with pd.ExcelWriter(log_path, engine="xlsxwriter") as xw:
    pd.DataFrame({"PDI": moved}).to_excel(xw, index=False, sheet_name="moved")
    pd.DataFrame({"PDI": skipped_exists}).to_excel(xw, index=False, sheet_name="skipped_exists")
    pd.DataFrame({"PDI": not_found}).to_excel(xw, index=False, sheet_name="not_found")
    if errors:
        pd.DataFrame(errors, columns=["PDI", "error"]).to_excel(xw, index=False, sheet_name="errors")

print(f"\n[OK] Log saved: {log_path}")
print(f"[INFO] Files moved to: {USED}")

Moving selected subjects: 100%|█████████████████████████████████████████████████| 102/102 [00:00<00:00, 144.22person/s]


=== SUMMARY ===
Total in Excel (unique PDI): 102
Matched in source: 101/102 (99.0%)
Matched & moved this run: 101/102 (99.0%)
Already in USED (skipped): 0
Not found in source: 1
Errors: 0

[OK] Log saved: D:\JHAhn\8641_conductivity\DTI_ALPS\1028\USED_move_log.xlsx
[INFO] Files moved to: D:\JHAhn\8641_conductivity\DTI_ALPS\1028\8641_Dem+MCI+AD\USED


In [11]:
unmatched = sorted(set(not_found))  # 중복 제거 + 정렬
print(f"[UNMATCHED] {len(unmatched)}명")
for name in unmatched:
    print(name)

[UNMATCHED] 1명
HFC2021AD29_LBD_NODDI32


In [ ]:
## ALPS Results to dF

In [ ]:
COLS = [
    "PDI",
    "LT_ALL_Dxx_proj","LT_ALL_Dyy_proj","LT_ALL_Dzz_proj","LT_ALL_Dxx_assoc","LT_ALL_Dyy_assoc","LT_ALL_Dzz_assoc","LT_ALL_ALPS index",
    "LT_b800_Dxx_proj","LT_b800_Dyy_proj","LT_b800_Dzz_proj","LT_b800_Dxx_assoc","LT_b800_Dyy_assoc","LT_b800_Dzz_assoc","LT_b800_ALPS index",
    "LT_b2000_Dxx_proj","LT_b2000_Dyy_proj","LT_b2000_Dzz_proj","LT_b2000_Dxx_assoc","LT_b2000_Dyy_assoc","LT_b2000_Dzz_assoc","LT_b2000_ALPS index",
    "RT_ALL_Dxx_proj","RT_ALL_Dyy_proj","RT_ALL_Dzz_proj","RT_ALL_Dxx_assoc","RT_ALL_Dyy_assoc","RT_ALL_Dzz_assoc","RT_ALL_ALPS index",
    "RT_b800_Dxx_proj","RT_b800_Dyy_proj","RT_b800_Dzz_proj","RT_b800_Dxx_assoc","RT_b800_Dyy_assoc","RT_b800_Dzz_assoc","RT_b800_ALPS index",
    "RT_b2000_Dxx_proj","RT_b2000_Dyy_proj","RT_b2000_Dzz_proj","RT_b2000_Dxx_assoc","RT_b2000_Dyy_assoc","RT_b2000_Dzz_assoc","RT_b2000_ALPS index",
    "LT_CTI_Proj_Dxx","LT_CTI_Proj_Dyy","LT_CTI_Proj_Dzz","LT_CTI_Assoc_Dxx","LT_CTI_Assoc_Dyy","LT_CTI_Assoc_Dzz","LT_CTI-ALPS index",
    "RT_CTI_Proj_Dxx","RT_CTI_Proj_Dyy","RT_CTI_Proj_Dzz","RT_CTI_Assoc_Dxx","RT_CTI_Assoc_Dyy","RT_CTI_Assoc_Dzz","RT_CTI-ALPS index",
]

MAP = {
    "Auto_ALPS_index_LH_MNIspace.csv": [
        "LT_ALL_Dxx_proj","LT_ALL_Dyy_proj","LT_ALL_Dzz_proj","LT_ALL_Dxx_assoc","LT_ALL_Dyy_assoc","LT_ALL_Dzz_assoc","LT_ALL_ALPS index"
    ],
    "Auto_ALPS_index_LH_maxbval800_MNIspace.csv": [
        "LT_b800_Dxx_proj","LT_b800_Dyy_proj","LT_b800_Dzz_proj","LT_b800_Dxx_assoc","LT_b800_Dyy_assoc","LT_b800_Dzz_assoc","LT_b800_ALPS index"
    ],
    "Auto_ALPS_index_LH_maxbval2000_MNIspace.csv": [
        "LT_b2000_Dxx_proj","LT_b2000_Dyy_proj","LT_b2000_Dzz_proj","LT_b2000_Dxx_assoc","LT_b2000_Dyy_assoc","LT_b2000_Dzz_assoc","LT_b2000_ALPS index"
    ],
    "Auto_ALPS_index_RH_MNIspace.csv": [
        "RT_ALL_Dxx_proj","RT_ALL_Dyy_proj","RT_ALL_Dzz_proj","RT_ALL_Dxx_assoc","RT_ALL_Dyy_assoc","RT_ALL_Dzz_assoc","RT_ALL_ALPS index"
    ],
    "Auto_ALPS_index_RH_maxbval800_MNIspace.csv": [
        "RT_b800_Dxx_proj","RT_b800_Dyy_proj","RT_b800_Dzz_proj","RT_b800_Dxx_assoc","RT_b800_Dyy_assoc","RT_b800_Dzz_assoc","RT_b800_ALPS index"
    ],
    "Auto_ALPS_index_RH_maxbval2000_MNIspace.csv": [
        "RT_b2000_Dxx_proj","RT_b2000_Dyy_proj","RT_b2000_Dzz_proj","RT_b2000_Dxx_assoc","RT_b2000_Dyy_assoc","RT_b2000_Dzz_assoc","RT_b2000_ALPS index"
    ],
    "Auto_CTI_ALPS_index_LH.csv": [
        "LT_CTI_Proj_Dxx","LT_CTI_Proj_Dyy","LT_CTI_Proj_Dzz","LT_CTI_Assoc_Dxx","LT_CTI_Assoc_Dyy","LT_CTI_Assoc_Dzz","LT_CTI-ALPS index"
    ],
    "Auto_CTI_ALPS_index_RH.csv": [
        "RT_CTI_Proj_Dxx","RT_CTI_Proj_Dyy","RT_CTI_Proj_Dzz","RT_CTI_Assoc_Dxx","RT_CTI_Assoc_Dyy","RT_CTI_Assoc_Dzz","RT_CTI-ALPS index"
    ],
}

In [ ]:
def parse_dti_csv(fp: str):
    try:
        df = pd.read_csv(fp)
        df.columns = [c.strip() for c in df.columns]
        keys = ["Dxx_proj","Dyy_proj","Dzz_proj","Dxx_assoc","Dyy_assoc","Dzz_assoc","ALPS index"]
        if all(k in df.columns for k in keys):
            row = df.iloc[0][keys]
            return [pd.to_numeric(x, errors="coerce") for x in row.tolist()]
    except Exception:
        pass
    # 폴백: 텍스트에서 숫자 7개 추출
    with open(fp, "r", errors="ignore") as f:
        txt = f.read()
    nums = re.findall(r"[-+]?\d*\.\d+|\d+", txt)
    return [float(x) for x in nums[:7]] if len(nums) >= 7 else [float("nan")]*7

def parse_cti_csv(fp: str):
    with open(fp, "r", errors="ignore") as f:
        txt = f.read()
    nums = re.findall(r"[-+]?\d*\.\d+|\d+", txt)
    return [float(x) for x in nums[:7]] if len(nums) >= 7 else [float("nan")]*7

def parse_file(fp: str, fname: str):
    return parse_cti_csv(fp) if "CTI" in fname else parse_dti_csv(fp)

# Collect individual folders 
subjects = []
if os.path.isdir(USED):
    group_dirs = [os.path.join(USED, g) for g in os.listdir(USED) if os.path.isdir(os.path.join(USED, g))]
    for g in group_dirs:
        for d in os.listdir(g):
            p = os.path.join(g, d)
            if os.path.isdir(p):
                subjects.append(p)
else:
    # (옵션) USED가 없으면 이전 방식으로 RT_dir 바로 아래 폴더를 개인으로 간주
    subjects = [os.path.join(RT_dir, d) for d in os.listdir(RT_dir) if os.path.isdir(os.path.join(RT_dir, d))]

subjects.sort(key=lambda p: os.path.basename(p))

rows = []
for subj in tqdm(subjects, desc="Subjects", unit="person"):
    row = {c: pd.NA for c in COLS}
    row["PDI"] = os.path.basename(subj)
    stats_dir = os.path.join(subj, "Results", "Stats")
    if os.path.isdir(stats_dir):
        for fname, target_cols in MAP.items():
            fp = os.path.join(stats_dir, fname)
            if not os.path.exists(fp):
                continue
            vals = parse_file(fp, fname)
            for c, v in zip(target_cols, vals):
                row[c] = v
    rows.append(row)

df = pd.DataFrame(rows)[COLS]
print(f"[INFO] subjects found: {len(subjects)}")
print(f"[INFO] df shape: {df.shape}")
df

In [ ]:
## SAVE df

# XLSX
xlsx_path = os.path.join(base, "Auto_ALPS_CTI_merged.xlsx")
with pd.ExcelWriter(xlsx_path, engine="xlsxwriter") as xw:
    df.to_excel(xw, index=False, sheet_name="merged")

print(f"[OK] Saved:\n - {csv_path}\n - {xlsx_path}")

In [ ]:
# Tissue Volumes (TiV) to dF

In [ ]:
group_dirs = [os.path.join(USED, g) for g in os.listdir(USED) if os.path.isdir(os.path.join(USED, g))]
subjects = []
for g in group_dirs:
    for d in os.listdir(g):
        p = os.path.join(g, d)
        if os.path.isdir(p):
            subjects.append(p)

# Sort(by folder names of each individual)
subjects.sort(key=lambda p: os.path.basename(p))

rows = []
cols_template = None
found, not_found = [], []

for subj in tqdm(subjects, desc="Collect BrainVolumeStats (USED/<group>/<person>)", unit="person"):
    pid_folder = os.path.basename(subj)
    fpath = os.path.join(subj, "Results", "Stats", "eTIV_NormalizedBrainVolumeStats.csv")
    if not os.path.exists(fpath):
        not_found.append(os.path.join(os.path.basename(os.path.dirname(subj)), pid_folder))  # group/person 표기
        continue

    df = pd.read_csv(fpath, sep=None, engine="python")
    df.columns = [str(c).strip() for c in df.columns]
    if "Measure:volume" not in df.columns:
        raise ValueError(f"[{pid_folder}] 'Measure:volume' 컬럼을 찾지 못했습니다: {fpath}")

    if cols_template is None:
        cols_template = [c for c in df.columns if c != "Measure:volume"]

    row = df.iloc[0].copy()
    pdi = str(row["Measure:volume"]).strip()
    row = row.drop(labels=["Measure:volume"]).apply(pd.to_numeric, errors="coerce")

    out = {"PDI": pdi}
    for c in cols_template:
        out[c] = row.get(c, pd.NA)
    rows.append(out)
    found.append(pid_folder)

# Merge-Summary-Save
merged = pd.DataFrame(rows, columns=["PDI"] + (cols_template or []))

print("\n=== SUMMARY ===")
print(f"Groups under USED: {len(group_dirs)}")
print(f"Persons discovered: {len(subjects)}")
print(f"Found BrainVolumeStats.csv: {len(found)}")
print(f"Merged rows: {len(merged)}")
print(f"Missing BrainVolumeStats.csv: {len(not_found)}")

if not_found:
    print("\n[UNMATCHED - no BrainVolumeStats.csv] (group/person)")
    for n in sorted(not_found):
        print(" -", n)

merged

In [ ]:
# # Save

# xlsx_out = os.path.join(base, "BrainVolumeStats_merged.xlsx")
# merged.to_csv(csv_out, index=False)
# with pd.ExcelWriter(xlsx_out, engine="xlsxwriter") as xw:
#     merged.to_excel(xw, index=False, sheet_name="merged")

# print(f"\n[OK] Saved:\n - {csv_out}\n - {xlsx_out}")